# Controlled Pendulum System — Dimensioning & Modeling Rationale

This notebook documents the **system engineering rationale** for the Controlled Pendulum demonstrator used in the SysSimX framework. It defines the system of interest, modeling scenarios, component choices, and a **dimensioning procedure** that links plant properties to actuator and controller capabilities.

**Purpose:** establish a consistent, professional basis for modeling, simulation, and thesis documentation.

## 1) System of Interest (SoI)

We model a **cyber‑physical controlled pendulum system** with the goal of trajectory tracking under different interaction scenarios.

**Core task:** track a prescribed angular reference trajectory using a feedback controller and electromechanical drive.

**Scenarios:**
- **Nominal tracking:** no contacts/impacts.
- **Contact scenario:** pendulum hits a wall (impact event), triggering a discrete reset (hybrid event handling).


## 2) Cyber & Physical Components

**Physical domain**
- Pendulum dynamics (rigid‑body plant)
- Actuator/drive (motor + gearbox)
- Optional wall/contact model

**Cyber domain**
- Reference generator (trajectory)
- Sensor chain (potentiometer + ADC)
- Decoder (ADC → angle)
- Controller (PID, continuous / sampled)
- Event logic (integrator reset on impact)


## 3) Component Modeling Levels

A key goal is **multi‑fidelity modeling**. The same conceptual component can exist at multiple levels:

**Drive (Actuator)**
- **Simple:** quasi‑static torque with limits
- **Dynamic:** electrical dynamics (L/R), current limit, viscous friction
- **Advanced:** motor inertia, compliance, gear, backlash (for feedthrough/hybrid studies)

**Plant (Pendulum)**
- **Modelica:** rigid‑body 1‑DOF reference model
- **OpenSim:** musculoskeletal‑style model
- **FEM / NGSolve:** deformable pendulum for contact/impact fidelity

**Controller (PID)**
- **Continuous:** continuous‑time PID with derivative filtering
- **Sampled:** discrete implementation for hybrid/co‑sim scenarios


## 4) Modeling Technologies

| Component | Modeling Approach | Example Implementation |
|---|---|---|
| Pendulum | Modelica | `ControlledPendulum.Plants.Pendulum` |
| Pendulum | OpenSim | `.osim` models |
| Pendulum | FEM | NGSolve + Python |
| Drive | Modelica | `DriveSimple`, `DriveDynamic`, `DriveAdvanced` |
| Sensors | Modelica | `AnglePotentiometerADC` + Decoder |
| Controller | Modelica | `PIDContinuous`, `PID_Sampled` |
| Integration | FMU | FMI 2.0 Co‑Simulation |


## 5) Signal Interfaces and Units

**Reference:** `θ_ref` [rad]  
**Measured angle:** `θ_meas` [rad]  
**Control command:** `u_cmd` [–] (normalized)  
**Motor torque:** `τ` [N·m]  
**Sensor voltage:** `v_out` [V]

This separation keeps control gains meaningful in physical units.

## 6) Dimensioning Procedure (Plant ↔ Drive Matching)

For a sinusoidal reference:
$$
\theta(t) = A \sin(\omega t),\quad \omega = 2\pi f
$$

**Max kinematics:**
- $|\dot{\theta}|_{max} = A\omega$
- $|\ddot{\theta}|_{max} = A\omega^2$

Plant equation:
$$
I \ddot{\theta} + m g L \sin(\theta) = \tau_{drive}
$$

**Required torque (conservative):**
$$
\tau_{req,max} \approx I A\omega^2 + m g L \sin(A)
$$

**Available torque (drive):**
$$
\tau_{max} \approx \eta \cdot I_G \cdot k_t \cdot I_{max}
$$

If $\tau_{max} < \tau_{req,max}$, tracking is infeasible without:
- reducing mass/inertia/length,
- reducing trajectory amplitude or frequency,
- increasing supply/current/gear ratio.


## 7) Baseline Parameter Set (to be filled)

| Parameter | Value | Notes |
|---|---:|---|
| Pendulum mass `m` |  |  |
| Pendulum length `L` |  |  |
| Pendulum inertia `I` |  |  |
| Reference amplitude `A` |  |  |
| Reference frequency `f` |  |  |
| Drive `I_max` |  |  |
| Drive `k_t` |  |  |
| Drive `gearRatio` |  |  |
| Drive `η` |  |  |


## 8) Simulation Scenarios

- **Nominal tracking:** no contact, stable continuous dynamics
- **Wall impact:** event‑driven reset and hybrid algorithm verification
- **Model‑fidelity comparison:** Modelica vs OpenSim vs FEM
- **Co‑simulation studies:** different master algorithms (Jacobi, GS, IJCSA)


## 9) Why simulation? (System engineering rationale)

- Test actuator sizing before building hardware
- Evaluate hybrid events (contact, resets, ZOH)
- Validate multi‑model consistency
- Compare numerical co‑simulation algorithms


In [1]:
# Optional quick sizing check (fill values)
import math

# Reference trajectory
A = 0.35        # rad
f = 0.25        # Hz
w = 2*math.pi*f

# Pendulum parameters
m = 5.0         # kg
L = 0.5         # m
I = 0.7         # kg*m^2
g = 9.81        # m/s^2

# Drive parameters
eta = 0.85
gear = 60
k_t = 0.03      # N*m/A
I_max = 10      # A

tau_req = I*A*w*w + m*g*L*math.sin(A)
tau_max = eta*gear*k_t*I_max

print("Required torque (approx):", round(tau_req, 3), "N*m")
print("Available torque:", round(tau_max, 3), "N*m")
print("Margin (tau_max / tau_req):", round(tau_max/tau_req, 2))


Required torque (approx): 9.014 N*m
Available torque: 15.3 N*m
Margin (tau_max / tau_req): 1.7
